# 03 — Hallucination Cases and Absent Concepts

The most theoretically interesting cases for the paper: languages where the
pipeline produces confident-sounding translations that may not reflect actual
community usage, or where the concept simply has no established equivalent.

This notebook reframes 'hallucination' as a productive analytical lens rather
than a pipeline failure. When an LLM produces a plausible-sounding translation
for a concept that has no community adoption, that is evidence about:
  - The distribution of DH discourse in LLM training data
  - Which communities' scholarly production is represented in those corpora
  - The relationship between language coverage and epistemic inclusion

**Run order:** After `evaluate_confidence.py`, `evaluate_llm_judge.py`,
and `evaluate_github_alignment.py`.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from data_generation_scripts.utils import get_data_directory_path, read_csv_file

DATA_DIR = get_data_directory_path()
EVAL_DIR = os.path.join(DATA_DIR, 'metadata_files', 'evaluation')

# Load all evaluation outputs
conf_df    = read_csv_file(os.path.join(EVAL_DIR, 'confidence_scores.csv'))\
             if os.path.exists(os.path.join(EVAL_DIR, 'confidence_scores.csv')) else None
judge_df   = read_csv_file(os.path.join(EVAL_DIR, 'llm_judge_scores.csv'))\
             if os.path.exists(os.path.join(EVAL_DIR, 'llm_judge_scores.csv')) else None
absent_df  = read_csv_file(os.path.join(EVAL_DIR, 'absent_concept_candidates.csv'))\
             if os.path.exists(os.path.join(EVAL_DIR, 'absent_concept_candidates.csv')) else None
align_df   = read_csv_file(os.path.join(EVAL_DIR, 'github_alignment.csv'))\
             if os.path.exists(os.path.join(EVAL_DIR, 'github_alignment.csv')) else None

for name, df in [('Confidence', conf_df), ('Judge', judge_df),
                  ('Absent concepts', absent_df), ('GitHub alignment', align_df)]:
    print(f'{name}: {len(df) if df is not None else "NOT FOUND"} rows')

## 3.1 Low-Confidence Cases: Primary Services Disagree

These rows are where primary services produced translations but couldn't agree.
This is distinct from 'absent concept' — the concept may exist in some form
in the community but there's no consensus on the right term.

In [ ]:
if conf_df is not None:
    low_conf = read_csv_file(os.path.join(EVAL_DIR, 'low_confidence.csv'))\
               if os.path.exists(os.path.join(EVAL_DIR, 'low_confidence.csv')) else None
    
    if low_conf is not None:
        print(f'Low-confidence rows: {len(low_conf)}')
        print(f'Distribution of confidence scores:')
        print(low_conf['primary_confidence'].describe().round(3).to_string())
        
        # By language family
        LANGUAGE_FAMILIES = {
            'Romance':['es','fr','it','pt','ro','ca'], 'Germanic':['de','nl','sv','da','no'],
            'Slavic':['ru','pl','cs','sk','bg','hr','sr','uk'], 'East Asian':['zh','ja','ko'],
            'Semitic':['ar','he'], 'South Asian':['hi','bn','ur','ta','te'],
        }
        def get_family(code):
            for f, codes in LANGUAGE_FAMILIES.items():
                if code in codes: return f
            return 'Other'
        
        low_conf['family'] = low_conf['language_code'].apply(get_family)
        family_counts = low_conf['family'].value_counts()
        print('\nLow-confidence cases by language family:')
        print(family_counts.to_string())

## 3.2 Absent Concept Candidates

Languages where the pipeline produced translations but GitHub data shows
no community usage. These are the hallucination candidates — the most
analytically important rows for the paper.

In [ ]:
if absent_df is not None and len(absent_df) > 0:
    print(f'Absent concept candidates: {len(absent_df)} languages')
    
    # Show the translations that were produced with no GitHub signal
    variant_cols = [c for c in absent_df.columns if c.endswith('_term') and not c.endswith('_signal')]
    display_cols = ['language_code', 'language_name'] + variant_cols[:3]
    available = [c for c in display_cols if c in absent_df.columns]
    print('\nSample of absent-concept translations produced by pipeline:')
    print(absent_df[available].head(20).to_string(index=False))
    
    # Cross-reference with judge's concept_exists assessments
    if judge_df is not None and 'concept_exists' in judge_df.columns:
        judge_absent = judge_df[judge_df['concept_exists'] == False]
        print(f'\nRows where LLM judge also flagged concept as absent: {len(judge_absent)}')
        
        # Find overlap
        if 'language_code' in absent_df.columns and 'language_code' in judge_absent.columns:
            overlap_codes = set(absent_df['language_code']) & set(judge_absent['language_code'])
            print(f'Overlap (both GitHub and judge flag as absent): {len(overlap_codes)} languages')
            print(sorted(overlap_codes))
else:
    print('No absent concept candidates found (or github_alignment.py not yet run)')

## 3.3 What Did the Pipeline Confabulate?

For languages flagged as absent-concept candidates, examine what translations
were produced. Classify them:
- **Transliteration**: borrowed English sounds (e.g. "Kompyuterasyon Humaniti")
- **Calque**: word-for-word structural translation
- **Neologism**: new compound using native roots
- **Loan word**: English term unchanged

In [ ]:
if absent_df is not None:
    # Compare how similar each translation is to the English source term
    # A high similarity may indicate a loan word or transliteration
    
    def levenshtein_sim(a, b):
        if not a or not b: return 0.0
        a, b = a.lower(), b.lower()
        if a == b: return 1.0
        m, n = len(a), len(b)
        dp = list(range(n+1))
        for i in range(1, m+1):
            prev = dp[:]
            dp[0] = i
            for j in range(1, n+1):
                dp[j] = prev[j-1] if a[i-1]==b[j-1] else 1+min(prev[j-1],prev[j],dp[j-1])
        return 1.0 - dp[n]/max(m,n)
    
    term = 'Computational Humanities'
    absent_term = absent_df[absent_df['term_source'] == term] if 'term_source' in absent_df.columns else absent_df
    
    if len(absent_term) > 0:
        variant_cols = [c for c in absent_term.columns if c.endswith('_term') and
                        not c.endswith('_signal') and not c.endswith('_evidence')]
        
        for col in variant_cols[:2]:  # first two variants
            vals = absent_term[['language_code', 'language_name', col]].dropna(subset=[col])
            vals['sim_to_english'] = vals[col].apply(
                lambda t: levenshtein_sim(str(t), term)
            )
            vals['type'] = vals['sim_to_english'].apply(
                lambda s: 'loan/transliteration' if s > 0.6 else
                          'close calque' if s > 0.4 else 'native term'
            )
            print(f'\n{col} — translation types for absent-concept languages:')
            print(vals['type'].value_counts().to_string())
            print('\nSamples:')
            for t in ['loan/transliteration', 'close calque', 'native term']:
                subset = vals[vals['type'] == t].head(3)
                for _, r in subset.iterrows():
                    lang = r.get('language_name', r.get('language_code', '?'))
                    print(f'  [{t}] {lang}: {r[col]} (sim={r["sim_to_english"]:.2f})')

## 3.4 Judge Reasoning for Absent Concepts

Read the LLM judge's reasoning for cases where it flagged the concept as absent.
This qualitative data is valuable for the paper.

In [ ]:
if judge_df is not None and 'concept_exists' in judge_df.columns:
    absent_judge = judge_df[
        (judge_df['concept_exists'] == False) |
        (judge_df['concept_exists'] == 'False')
    ].copy()
    
    print(f'Judge flagged {len(absent_judge)} rows as absent concept')
    print(f'\nExistence notes from judge:')
    for _, row in absent_judge.head(10).iterrows():
        lang = row.get('language_name', row.get('language_code', '?'))
        notes = row.get('existence_notes', '')
        if notes:
            print(f'  [{lang}]: {notes[:200]}')
    
    # Score distribution for absent vs. present concepts
    judge_df['concept_binary'] = judge_df['concept_exists'].map(
        {True: 'exists', False: 'absent', 'True': 'exists', 'False': 'absent'}
    ).fillna('unknown')
    
    fig, ax = plt.subplots(figsize=(8, 4))
    for label, colour in [('exists', 'steelblue'), ('absent', 'firebrick'), ('unknown', 'grey')]:
        subset = judge_df[judge_df['concept_binary'] == label]['best_score'].dropna()
        if len(subset): ax.hist(subset, bins=20, alpha=0.6, label=label, color=colour)
    ax.set_xlabel('Judge Best Score')
    ax.set_ylabel('N rows')
    ax.set_title('Judge Score Distribution: Concept Present vs. Absent')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, '03_absent_concept_scores.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Judge scores not available or missing concept_exists column')

## 3.5 Loan Word Communities

Communities that use the English term directly — this is not a translation failure.
It is evidence that the community has chosen to adopt the English term rather
than construct a native equivalent, which is a meaningful scholarly choice.

In [ ]:
loan_path = os.path.join(EVAL_DIR, 'loan_word_cases.csv')
if os.path.exists(loan_path):
    loan_df = read_csv_file(loan_path)
    print(f'Loan word communities: {len(loan_df)} languages')
    
    lang_col = 'language_name' if 'language_name' in loan_df.columns else 'language_code'
    if lang_col in loan_df.columns:
        print('Languages where GitHub data shows English term in use:')
        for _, row in loan_df.iterrows():
            print(f'  {row["language_code"]:5s} {row.get(lang_col, "")}')
else:
    print('Loan word cases file not found — run evaluate_github_alignment.py')